# Null Handling, Missing Data Strategies, and Quality Validation

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 07_nulls_and_data_quality.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark handles null and NaN values, using na.fill(), na.drop(), and na.replace(), null-safe equality (<=>), and building validation pipelines.
- **Why does this topic exist?**: Real-world data is inherently dirty. Missing keys break joins, skew partitions, and corrupt aggregations.
- **What problem does it solve?**: Silent null propagation causes misleading report totals or runtime NullPointerExceptions.
- **Where is it used in real-world Data Engineering?**: Auditing data quality gates in production Lakehouse architectures.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrames, SQL, joins |
| **Required depth** | Working knowledge: can handle nulls, validate data, and quarantine bad records |
| **Data Engineer** | Use null-safe expressions, validation rules, and quarantine patterns. |
| **Senior Data Engineer** | Design data-quality gates with metrics and idempotent remediation. |
| **Spark specialist** | Reason about three-valued logic, ANSI behavior, and data contracts. |
| **Next topic** | `03_dataframe_and_spark_sql/08_udfs_and_pandas_udfs.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Nulls are not just missing values; they change join behavior, aggregation results, and filter semantics.
`col == None` is not a valid null check, `sum` ignores nulls, and an inner join drops rows with null keys. Most
production data-quality incidents are not exotic; they are ordinary null and duplicate problems that were never
validated.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Null** | Unknown/missing value | SQL three-valued logic; not equal to anything |
| **NaN** | Not-a-number (floating point) | Different from null |
| **`isNull`/`isNotNull`** | Null checks | Use instead of `== None` |
| **`coalesce`** | First non-null | Useful for defaults |
| **Quarantine** | Separate bad records | Preserve for audit/reprocessing |
| **Data quality gate** | Validation before publish | Counts, ranges, uniqueness, referential checks |

### How it works, step by step

1. The reader may produce nulls for missing fields or malformed values.
2. Filters evaluate three-valued logic; `null` is not true, so the row is dropped.
3. Joins treat null keys as non-matching in equi-joins.
4. Aggregations ignore nulls by default (except `count(*)`).
5. Validation rules classify rows as good or quarantined.
6. Metrics are written for monitoring and alerting.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Null Semantics in Spark SQL

```
+-------------------------------------------------------------------------------+
| Spark SQL Null Handling Rules                                                 |
+-------------------------------------------------------------------------------+
| Standard Equality (col == null)     | Always returns NULL (unknown)           |
| Null-Safe Equality (col <=> null)   | Returns TRUE if both null, FALSE if one |
| Standard Aggregations (SUM, AVG)    | Silently ignores NULL values            |
| COUNT(*)                            | Counts ALL rows including nulls         |
| COUNT(col)                          | Counts only NON-NULL rows               |
+-------------------------------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Quality_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Auditing Missing Data in `/opt/spark-data/quality/orders_dirty.csv`

In [ ]:
dirty_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/quality/orders_dirty.csv")

print(f"Total rows in dirty dataset: {dirty_df.count()}")
dirty_df.printSchema()

# Profile null counts across all columns
null_counts = dirty_df.select([
    F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), c)).alias(c)
    for c in dirty_df.columns
])

print("Null counts per column:")
null_counts.show()

## 4. Cleaning Strategies: `na.fill()`, `na.drop()`, and `coalesce()`

In [ ]:
# Strategy 1: Drop rows where critical primary keys are null
cleaned_step1 = dirty_df.na.drop(subset=["order_id", "customer_id"])

# Strategy 2: Fill null discounts with 0.0 and null status with 'Pending'
cleaned_step2 = cleaned_step1.na.fill({
    "discount": 0.0,
    "status": "Pending",
    "channel": "Unknown"
})

print(f"Rows before: {dirty_df.count()} -> Rows after key drop: {cleaned_step1.count()}")
cleaned_step2.filter("discount IS NULL OR status IS NULL").show()
print("Cleaned sample:")
cleaned_step2.show(5)

---

## Experiment: Null-Safe Equality (<=>) in Filter and Joins

- **1. What are we testing?**: Demonstrate that standard'=='evaluates to null when comparing null values, whereas'<=>'evaluates deterministically.
- **2. What do we expect?**: df.filter(col('a') == col('b')) drops rows where both are null; col('a').eqNullSafe(col('b')) retains them.

In [ ]:
test_nulls = spark.createDataFrame([
    (1, "A", "A"),
    (2, None, None),
    (3, "B", None)
], ["id", "val1", "val2"])

print("Standard Equality (val1 == val2):")
test_nulls.filter(F.col("val1") == F.col("val2")).show()

print("Null-Safe Equality (val1.eqNullSafe(val2)):")
test_nulls.filter(F.col("val1").eqNullSafe(F.col("val2"))).show()

**3. What happened?**: Standard equality dropped row 2 (both None); null-safe equality correctly preserved row 2.  
**4. Why did it happen?**: In ANSI SQL three-valued logic, NULL == NULL produces NULL (falsy in filters). eqNullSafe explicitly tests for equivalence.  
**5. Practical Engineering Lesson**: Always use eqNullSafe() when joining or filtering nullable keys where null-to-null matching is required.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Customer data quality for CRM

- **Industry**: Retail
- **Scenario**: A CRM ingests customer records from web, stores, and partners.
- **Problem**: Duplicate emails, missing phone numbers, and invalid countries corrupt segmentation.
- **Why this concept is relevant**: Null handling and validation rules prevent bad records from reaching the golden customer table.
- **Implementation idea**: Normalize, deduplicate by a deterministic key, validate required fields, quarantine failures, and publish metrics.
- **Expected result**: A trusted customer dimension plus a quarantine dataset.
- **Alternative**: A master data management platform.
- **Trade-offs**: Spark handles batch remediation; MDM provides ongoing stewardship and matching.

### Use Case 2: Financial reconciliation controls

- **Industry**: Banking
- **Scenario**: A finance team reconciles transactions against ledger entries.
- **Problem**: A single null amount or duplicate transaction breaks the reconciliation and delays reporting.
- **Why this concept is relevant**: Explicit null checks, uniqueness tests, and amount-range validation catch issues before publishing.
- **Implementation idea**: Run quality rules, compare counts/sums, quarantine failures, and block publication on critical violations.
- **Expected result**: An auditable reconciliation with a clear failure reason.
- **Alternative**: A warehouse with dbt tests and constraints.
- **Trade-offs**: Spark is flexible for large feeds; dbt/warehouse tests are simpler for curated SQL models.

---

## Business Value

- Improves trust in reporting and decision-making.
- Reduces manual reconciliation and rework.
- Makes failures visible before they reach executives or regulators.
- Supports auditable quarantine and reprocessing.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Define required, unique, range, and referential rules explicitly.
- Write quality metrics and alert on trends, not just failures.
- Quarantine bad rows instead of silently dropping them.
- Use null-safe joins/expressions where appropriate.
- Make remediation idempotent and preserve raw data.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Null behavior in joins and aggregations

**Steps**

1. Create a small DataFrame with null keys and null values.
2. Join it to another DataFrame and count matched rows.
3. Run `sum`, `avg`, and `count` on a null-containing column.

**What to observe**: Null keys do not match in equi-joins; aggregations ignore nulls unless you handle them explicitly.

**Healthy result**: You can predict the row counts and aggregate results.

**Problematic result**: A join silently drops null-key rows, changing revenue totals.

**Engineering takeaway**: Treat null semantics as part of the business logic.

---

## Trade-offs and Alternatives

### Advantages

- Declarative null handling
- Scalable validation
- Quarantine supports audit

### Limitations

- Three-valued logic can surprise
- Rules need maintenance
- Validation adds compute

### When to use it

- Ingestion validation
- Financial reconciliation
- Customer data
- Healthcare records

### When not to use it

- Assuming null equals null
- Silently dropping bad rows
- Using NaN and null interchangeably

### Alternatives inside Spark

- `na` functions
- `coalesce`
- `dropDuplicates`
- ANSI mode

### Alternatives outside Spark

- dbt tests
- Great Expectations
- MDM platforms

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Filter Non-Null Valid Records
**Problem Statement**: Filter dirty_df to retain only rows where both quantity and unit_price are NOT null and greater than zero.  
**Input / Context**: dirty_df

In [ ]:
# Exercise 1: Valid numeric records

### Exercise 2 (Intermediate): Replace Invalid String Values with na.replace()
**Problem Statement**: Use na.replace() to standardize channel values: map'App'to'Mobile'and'N/A'to'Unknown'.  
**Input / Context**: dirty_df channel column

In [ ]:
# Exercise 2: na.replace mapping

### Exercise 3 (Advanced): Data Quality Metric Report Card
**Problem Statement**: Calculate the percentage completeness (non-null %) for every column in dirty_df and display as a summary DataFrame.  
**Input / Context**: dirty_df

In [ ]:
# Exercise 3: Completeness report

---

## Production Debugging Exercise

### The `== None` filter

**Scenario**: A filter `df.filter(F.col('amount') == None)` returns zero rows even though nulls exist.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Use `isNull()`.
2. Check the physical plan for the filter expression.
3. Remember SQL three-valued logic.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Filter Non-Null Valid Records):
```python
valid = dirty_df.filter(F.col("quantity").isNotNull() & (F.col("quantity") > 0) & F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))
print("Valid count:", valid.count())
```

#### Solution 2 (Replace Invalid String Values with na.replace()):
```python
standardized = dirty_df.na.replace({"App":"Mobile","N/A":"Unknown"}, subset=["channel"])
standardized.groupBy("channel").count().show()
```

#### Solution 3 (Data Quality Metric Report Card):
```python
total_rows = dirty_df.count()
completeness_exprs = [
    F.round((F.count(c) / total_rows) * 100, 2).alias(f"{c}_pct_complete")
    for c in dirty_df.columns
]
dirty_df.select(completeness_exprs).show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Null Semantics**: NULL == NULL produces NULL; use eqNullSafe() for null-aware matching.
- **na Package**: na.fill(), na.drop(), na.replace() provide high-level missing data management.
- **Aggregation Impact**: Standard aggregations ignore nulls, but COUNT(*) includes them.

### Essential Technical Terms
- `eqNullSafe`: Null-safe equality operator (<=>).
- `na.fill`: Replaces null values with specified constants.
- `na.drop`: Drops rows containing null values in designated subsets.

### Common Engineering Mistakes
- **Using == to check for null**: df.filter(col('x') == None) fails silently; must use col('x').isNull().
- **Forgetting that Joins Drop Null Keys**: Standard equi-joins drop rows with null keys on both sides.

### Interview Questions on This Topic
- **Q: What does COUNT(col) return if all values in that column are null?**
  - *A*: It returns 0.
- **Q: How does eqNullSafe differ from standard equality?**
  - *A*: It returns True when comparing NULL to NULL.


### More Interview Questions

- **Q: How do you check for nulls in Spark?**
  - *A*: Use `isNull()`/`isNotNull()` or `isnan()` for floating-point NaN; `== None` does not work as expected.
- **Q: What is the difference between `dropDuplicates` and `distinct`?**
  - *A*: `distinct` compares all columns; `dropDuplicates` can compare a subset of columns, but the retained row is non-deterministic unless you define an ordering.

### What to Study Next
Open 03_dataframe_and_spark_sql/08_udfs_and_pandas_udfs.ipynb to compare standard UDFs with Arrow Pandas UDFs.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')